<header>
   <p  style='font-size:36px;font-family:Arial; color:#F0F0F0; background-color: #00233c; padding-left: 20pt; padding-top: 20pt;padding-bottom: 10pt; padding-right: 20pt;'>
       Simplify Text Analytics with Teradata's Generative AI functions
  <br>
       <img id="teradata-logo" src="https://storage.googleapis.com/clearscape_analytics_demo_data/DEMO_Logo/teradata.svg" alt="Teradata" style="width: 125px; height: auto; margin-top: 20pt;">
    </p>
</header>

<p style="font-size:16px;font-family:Arial">Modern data teams need AI capabilities where their data lives. Teradata's integration with AWS Bedrock enables end users, data engineers, and data scientists to apply large language models directly within their existing tools and pipelines, with enterprise security and governance built in.

<div style="text-align: center;">
  <img src="./images/td_genai.png" 
       alt="otf" 
       style="width:100%; border: 4px solid #404040; border-radius: 10px;" />
</div><br>

<p style="font-size:16px;font-family:Arial">
Teradata's in-database GenAI operators bring Bedrock's powerful language models directly into your data workflows. .
Teradata coordinates LLM processing and data operations within the database itself. Bedrock credentials are securely managed using classical RBAC policies, while GenAI operators seamlessly integrate into existing SQL queries and data pipelines.

<p style="font-size:18px;font-family:Arial"><b>Analyzing Customer Complaints with GenAI</b></p>
<p style="font-size:16px;font-family:Arial">
Traditionally, performing sentiment analysis, summarization, and translation on customer complaints means <b>extracting data, calling multiple external APIs, managing credentials, handling errors, and loading results back </b> — a complex pipeline with significant data movement and latency.<br><b>With Teradata's GenAI operators, the entire workflow becomes a simple SQL query</b>. Sentiment, summarization, and translation happen in-database using AWS Bedrock models, eliminating data movement while maintaining enterprise security. Here's how it works.

<hr style="height:2px;border:none;">
<p style = 'font-size:20px;font-family:Arial'><b>1. Connect to Vantage</b></p>
<p style = 'font-size:16px;font-family:Arial'>First we will create the variable for magic command to connect to enviroment in the sql kernel. Hence we will remove any variable if any same name variable is already there. If it is not present below command will give error, please ignore the message "Profile does not exist".

In [ ]:
%rmconnect lake

<p style = 'font-size:16px;font-family:Arial'> For the below command please add the username given in the .env file without quotes. To get this open a new Terminal and locate the .env file we have provided.
Then select File > New Launcher > Terminal<br>
<img src="./images/terminal.png" alt="terminal" style="width: 50%; border: 4px solid #404040; border-radius: 10px;"/><br>
<br>
At the command prompt in the terminal, execute this command <code>cat ~/JupyterLabRoot/TeradataCloud/.config/.env</code><br>
The output from the command will be similar to this:<br>
<img src="./images/env_file.png" alt="env file" style="width: 60%; border: 4px solid #404040; border-radius: 10px;"/>     </li>

In [ ]:
%addconnect name=VCL, host=54.156.178.22, user=demo_20may_99u4m2v5sd495c4s

In [ ]:
%connect VCL

<hr style="height:2px;border:none;">
<p style = 'font-size:20px;font-family:Arial'><b>2. CSP Credentials</b></p>
<p style = 'font-size:16px;font-family:Arial'>Create an authorization object to store the user and secrets with access to the LLM service you want to use.In this demo, we will use AWS.<br>
<i>*You need to fill in AWS Access Key and AWS Secret Key in below sql to create authorization object

In [ ]:
CREATE AUTHORIZATION td_aws_genai_auth
       user '<aws access key>'
       password '<aws secret key>'

<p style = 'font-size:16px;font-family:Arial'>We can use this along model invocation arguments to call a LLM on bedrock:
<code style="padding:0; line-height:1.1;">
AWS_Bedrock_CSP_LLM_arguments
(
    ApiType('aws')
    [AUTHORIZATION([DatabaseName.]AuthorizationObjectName] 
    |{
    AcessKey('&lt;aws access key&gt;')
    SecretKey('&lt;aws secret key&gt;')|
    [SessionKey('&lt;aws_session_key&gt;')]
    }
    Region('&lt;aws region&gt;')
    ModelName('&lt;bedrock-model-name&gt;')
    [ModelArgs('{"Top_P":"top_p","Top_K":"top_k"}')]
)
    </code>

<hr style="height:2px;border:none;">
<p style = 'font-size:20px;font-family:Arial'><b>2. TextAnalyticsAI Functions</b></p>
<p style = 'font-size:16px;font-family:Arial'>The Teradata AI Text analytics functions provides over 11 built-in generative AI functions for powerful in-database NLP capabilities leveraging external LLM prividers:
<ul style = 'font-size:16px;font-family:Arial'>
<li><b>AI_Classify()</b> – Classify text into predefined categories                </li>
<li><b>AI_AnalyzeSentiment()</b> – Perform sentiment analysis                      </li>
<li><b>AI_DetectLanguage()</b> – Detect the language of a text                     </li>
<li><b>AI_Embeddings()</b> – Generate embeddings for similarity search             </li>
<li><b>AI_RecognizeEntities()</b> – Extract named entities                         </li>
<li><b>AI_RecognizePiiEntities()</b> – Detect and label PII entities               </li>
<li><b>AI_ExtractKeyPhrases()</b> – Identify key phrases in text                   </li>
<li><b>AI_MaskPii()</b> – Mask personally identifiable information (PII)           </li>
<li><b>AI_SentenceSimilarity()</b> – Measure semantic similarity between sentences </li>
<li><b>AI_Summarize()</b> – Generate summaries of longer documents                 </li>
<li><b>AI_Translate()</b> – Translate text between languages                       </li>
</ul>
<hr style="height:1px;border:none;">
<p style = 'font-size:18px;font-family:Arial;'><b>2.1 Data Overview</b><br>


In [ ]:
sel top 10 * from "DEMO_ComplaintAnalysis"."Consumer_Complaints" 

<hr style="height:1px;border:none;">
<p style = 'font-size:18px;font-family:Arial;'><b>2.2 Sentiment Extraction</b><br>
    <p style = 'font-size:16px;font-family:Arial;'>Generally, the sentiment expressed in complains is negative, let's check using the <b>AI_AnalyzeSentiment</b> function to be sure:</p>

In [ ]:
SELECT date_received, product, consumer_complaint_narrative, Sentiment
FROM AI_AnalyzeSentiment(
    ON (
        SELECT * FROM "DEMO_ComplaintAnalysis"."Consumer_Complaints" 
        QUALIFY ROW_NUMBER() OVER (ORDER BY "date_received")<11
    ) AS InputTable
    USING
        Authorization(td_aws_genai_auth)
        ModelName('us.amazon.nova-lite-v1:0')
        ApiType('aws')
        Region('us-east-1')
        TextColumn('consumer_complaint_narrative')
) AS sqlmr

<hr style="height:1px;border:none;">
<p style = 'font-size:18px;font-family:Arial;'><b>2.3 Generating Summary</b><br>
    <p style = 'font-size:16px;font-family:Arial;'>We can use the LLM co-processor to summarize content using the  <b>AI_TextSummarize</b> function:</p>

In [ ]:
create table complaints_summary as
(
    
SELECT Complaint_id, "Summary" Summary_txt, consumer_complaint_narrative Original_txt
FROM AI_TextSummarize(
    ON (
        SELECT * FROM "DEMO_ComplaintAnalysis"."Consumer_Complaints" 
        QUALIFY ROW_NUMBER() OVER (ORDER BY "date_received")<11
    ) AS InputTable
    USING
        Authorization(td_aws_genai_auth)
        ModelName('us.amazon.nova-lite-v1:0')
        ApiType('aws')
        Region('us-east-1')
        TextColumn('consumer_complaint_narrative')
) AS sqlmr

) with data;

In [ ]:
sel * from complaints_summary;

<hr style="height:1px;border:none;">
<p style = 'font-size:18px;font-family:Arial;'><b>2.4 Translating Content</b><br>
    <p style = 'font-size:16px;font-family:Arial;'>We can also translate content between languages using the <b>AI_TextTranslate</b> function:</p>

In [ ]:
SELECT *
FROM AI_TextTranslate(
    ON (
        SELECT * FROM complaints_summary
    ) AS InputTable
    USING
        Authorization(td_aws_genai_auth)
        ModelName('us.amazon.nova-lite-v1:0')
        ApiType('aws')
        Region('us-east-1')
        TextColumn('Summary_txt')
        TargetLang('portugese')
) AS sqlmr

<hr style="height:1px;border:none;">
<p style = 'font-size:18px;font-family:Arial;'><b>2.5 Translating content</b><br>
<p style = 'font-size:16px;font-family:Arial;'>The <b>AI_AskLLM</b> function is used to ask questions to the LLM based on the given context.<br>In this example, we wil pass two unrelated questions to our LLM with two contextual dataset and compute all the answers in batch.

In [ ]:
CREATE MULTISET TABLE question (
    id int, 
    text_data varchar(500)
);

In [ ]:
CREATE MULTISET TABLE context_data (
    id int, 
    text_data varchar(500)
);

In [ ]:
del from context_data;
del from question;

--Questions
INSERT INTO question VALUES(1, 'What is the top-performing store location mentioned?');
INSERT INTO question VALUES (2, 'Which product category generated the highest sales revenue?');
INSERT INTO question VALUES (3, 'Which loyalty program tier has the most members?');
INSERT INTO question VALUES (4, 'Which supplier delivered the highest percentage of on-time shipments?');


--Context data
INSERT INTO context_data (1, 'The flagship store in Chicago achieved record-breaking growth this quarter and is known for its high foot traffic.');
INSERT INTO context_data (1, 'The Chicago branch features an experiential layout and is famous for its modern automated checkout systems.');

INSERT INTO context_data (2, 'The Electronics department generated $40,000 in sales, which includes smartphone and accessory revenue.');
INSERT INTO context_data (2, 'The Apparel department brought in gross sales worth $20,000.');
INSERT INTO context_data (2, 'The Home Goods section recorded total transactions valued at $20,000.');


INSERT INTO context_data (3, 'The Gold Tier has 40,000 active members, which includes both online and in-store shoppers.');
INSERT INTO context_data (3, 'The Platinum Tier accounts for 20,000 registered customers.');
INSERT INTO context_data (3, 'The Diamond Tier requires premium invites and has a base valued at 20,000 users.');


INSERT INTO context_data (4, 'Apex Logistics achieved a 95% on-time delivery rate, which includes their regional and national routes.');
INSERT INTO context_data (4, 'Beacon Freight maintained a punctual delivery record worth 80% of total shipments.');
INSERT INTO context_data (4, 'Caliber Shipping completed their seasonal distribution schedule valued at an 80% success rate.');


In [ ]:
SELECT * FROM AI_AskLLM( 
      ON question AS InputTable partition by id
      ON context_data AS ContextTable partition by id
      USING   
      TextColumn('text_data')
      ApiType('aws')
      REGION('us-east-1')
      Authorization(td_aws_genai_auth)
      ModelName('us.amazon.nova-micro-v1:0')
      ContextColumn('text_data')
      Prompt('Provide clear and concise answer to this question:\n #QUESTION#\n\n Use this context data:\n #DATA#')
      DATAPOSITION('#DATA#')
      QUESTIONPOSITION('#QUESTION#')
      Accumulate('[0:]')
    ) as dt
    order by 1

 <p style = 'font-size:16px;font-family:Arial;'>Compare the answer quality with different models:</p>

In [ ]:
SELECT * FROM AI_AskLLM( 
      ON question AS InputTable partition by id
      ON context_data AS ContextTable partition by id
      USING   
      TextColumn('text_data')
      ApiType('aws')
      REGION('us-east-1')
      Authorization(td_aws_genai_auth)
      ModelName('us.anthropic.claude-sonnet-4-5-20250929-v1:0')
      MODELOPERATION('converse')
      ContextColumn('text_data')
      Prompt('Provide clear and concise answer to this question:\n #QUESTION#\n\n Use this context data:\n #DATA#')
      DATAPOSITION('#DATA#')
      QUESTIONPOSITION('#QUESTION#')
      isDebug('true')
      Accumulate('[0:]')
    ) as dt
    order by 1

<hr style="height:2px;border:none;">
<p style = 'font-size:20px;font-family:Arial'><b>3. Clean up</b></p>


In [ ]:
drop table complaints_summary;

In [ ]:
drop table question;

In [ ]:
drop table context_data;

<footer style="padding-bottom:35px; border-bottom:3px solid">
  <div style="float:right; margin-top:14px">Copyright © Teradata - 2026. All Rights Reserved</div>
</footer>